In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
import numpy as np
from sklearn.metrics import classification_report

In [3]:
# Set parameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
DATASET_PATH = 'D:\soyabean\dataset'

In [4]:
# Data augmentation and normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 616 images belonging to 7 classes.
Found 154 images belonging to 7 classes.


In [5]:
# Load base MobileNetV2 model
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze for initial training

# Build classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)


In [6]:

# Compile
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [8]:
# Train
model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)


Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 97s 5s/step - accuracy: 0.4989 - loss: 1.2525 - val_accuracy: 0.6104 - val_loss: 0.9522
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 98s 5s/step - accuracy: 0.6007 - loss: 1.1027 - val_accuracy: 0.6883 - val_loss: 0.8457
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 111s 6s/step - accuracy: 0.6531 - loss: 0.9347 - val_accuracy: 0.6623 - val_loss: 0.7453
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 105s 5s/step - accuracy: 0.6378 - loss: 0.8464 - val_accuracy: 0.6299 - val_loss: 0.8287
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 106s 5s/step - accuracy: 0.6475 - loss: 0.8527 - val_accuracy: 0.7013 - val_loss: 0.7431
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 136s 7s/step - accuracy: 0.6956 - loss: 0.7736 - val_accuracy: 0.6948 - val_loss: 0.7552
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 106s 5s/step - accuracy: 0.7534 - loss: 0.6775 - val_accuracy: 0.6883 - val_loss: 0.7544
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 132s 7s/step - accuracy: 0.7352 - loss: 0.6863 - val_accuracy: 0.6818 - val

In [9]:
# Optional: Fine-tune base model
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_generator, validation_data=val_generator, epochs=5)


Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 150s 7s/step - accuracy: 0.5490 - loss: 1.1816 - val_accuracy: 0.7143 - val_loss: 0.7400
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 125s 6s/step - accuracy: 0.5984 - loss: 1.0468 - val_accuracy: 0.7532 - val_loss: 0.7303
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 124s 6s/step - accuracy: 0.6947 - loss: 0.7991 - val_accuracy: 0.7078 - val_loss: 0.8189
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 128s 6s/step - accuracy: 0.6720 - loss: 0.8148 - val_accuracy: 0.7078 - val_loss: 0.7017
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 128s 6s/step - accuracy: 0.6806 - loss: 0.7813 - val_accuracy: 0.7403 - val_loss: 0.7907


In [10]:
  # Evaluate
val_generator.reset()
preds = model.predict(val_generator)
y_pred = np.argmax(preds, axis=1)
y_true = val_generator.classes

print(classification_report(y_true, y_pred, target_names=val_generator.class_indices.keys()))


5/5 ━━━━━━━━━━━━━━━━━━━━ 23s 5s/step
                       precision    recall  f1-score   support

    Bacterial Pustule       1.00      0.59      0.74        22
    Frogeye Leaf Spot       0.61      0.64      0.62        22
               Healty       0.79      1.00      0.88        22
                 Rust       0.33      0.23      0.27        22
Sudden Death Syndrome       0.71      0.91      0.80        22
     Target Leaf Spot       0.48      0.55      0.51        22
        Yellow Mosaic       0.95      0.95      0.95        22

             accuracy                           0.69       154
            macro avg       0.70      0.69      0.68       154
         weighted avg       0.70      0.69      0.68       154



In [ ]:
# Save Keras model
model.save("mobilenetv2_soybean_model.h5")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save TFLite model
with open("mobilenetv2_soybean_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpwuv10u40\assets


INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpwuv10u40\assets


Saved artifact at 'C:\Users\HP\AppData\Local\Temp\tmpwuv10u40'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  2333024592544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024589728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024588144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024589552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024591664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024598704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024597648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024594480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024590256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2333024598176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  23330245

In [45]:
import tensorflow as tf
import numpy as np
from PIL import Image

# --- Configuration ---
TFLITE_MODEL_PATH = "mobilenetv2_soybean_model.tflite"
CLASS_NAMES = [
    'Bacterial Pustule',
    'Frogeye Leaf Spot',
    'Healty',  # (Note: is this a typo? Should it be 'Healthy'?)
    'Rust',
    'Sudden Death Syndrome',
    'Target Leaf Spot',
    'Yellow Mosaic'
]
  # Modify if needed
IMG_SIZE = 224

# --- Load the TFLite model ---
print("📦 Loading TFLite model...")
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("✅ Model loaded.")

# --- Preprocess image ---
def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((IMG_SIZE, IMG_SIZE))
    image = np.array(image) / 255.0  # Normalize
    image = np.expand_dims(image, axis=0).astype(np.float32)  # Add batch dimension
    return image

# --- Predict ---
def predict(image_path):
    img = preprocess_image(image_path)
    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    predicted_class = np.argmax(output_data)
    confidence = np.max(output_data)
    return CLASS_NAMES[predicted_class], confidence

# --- Run Prediction Here ---
image_path = "WhatsApp Image 2025-04-29 at 6.55.45 PM.jpeg"  # 🖼️ UPDATE THIS with your actual image path
predicted_label, conf = predict(image_path)
print(f"🔍 Prediction: {predicted_label} (Confidence: {conf:.2f})")


📦 Loading TFLite model...
✅ Model loaded.
🔍 Prediction: Sudden Death Syndrome (Confidence: 0.60)


In [1]:
import tensorflow as tf

keras_model = tf.keras.models.load_model("mobilenetv2_soybean_model.h5")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
import tf2onxx

# Convert the Keras model to ONNX
spec = (tf.TensorSpec(keras_model.input.shape, tf.float32, name="input"),)
onnx_model, _ = tf2onnx.convert.from_keras(keras_model, input_signature=spec, opset=13)

# Save the ONNX model
with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())


ModuleNotFoundError: No module named 'tf2onxx'